In [1]:
import pandas as pd
import numpy as np
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'
print("All imports successful ✅")

All imports successful ✅


In [3]:
# Load clinical
clin = pd.read_csv(f'{base}/data/external/data_clinical_patient.txt',
                   sep='\t', skiprows=4)

# Load mRNA
mrna = pd.read_csv(f'{base}/data/external/data_mrna_seq_tpm.txt',
                   sep='\t', index_col=0)

print(f"Clinical shape: {clin.shape}")
print(f"mRNA shape:     {mrna.shape}")
print(f"\nVital status: {clin['VITAL_STATUS'].value_counts().to_dict()}")
print(f"Dead with survival time: {clin[clin['VITAL_STATUS']=='Dead']['DAYS_TO_DEATH'].notna().sum()}")
print(f"Alive with OS_MONTHS:    {clin[clin['VITAL_STATUS']=='Alive']['OS_MONTHS'].notna().sum()}")

# Max observed death time — used for censoring alive patients
max_death = clin[clin['VITAL_STATUS']=='Dead']['DAYS_TO_DEATH'].max()
print(f"\nMax observed death time: {max_death:.0f} days ({max_death/365:.1f} years)")
print(f"Alive patients will be censored at: {max_death:.0f} days")

Clinical shape: (230, 14)
mRNA shape:     (40796, 231)

Vital status: {'Alive': 152, 'Dead': 56}
Dead with survival time: 56
Alive with OS_MONTHS:    1

Max observed death time: 1703 days (4.7 years)
Alive patients will be censored at: 1703 days


In [4]:
# Build survival labels
# Dead patients: use DAYS_TO_DEATH as survival time, event=1
# Alive patients: censor at max observed death time (1703 days), event=0

survival_records = []

for _, row in clin.iterrows():
    patient_id = row['PATIENT_ID']
    vital = row['VITAL_STATUS']
    
    if vital == 'Dead' and pd.notna(row['DAYS_TO_DEATH']):
        survival_records.append({
            'patient_id': patient_id,
            'event': True,
            'time': float(row['DAYS_TO_DEATH'])
        })
    elif vital == 'Alive':
        survival_records.append({
            'patient_id': patient_id,
            'event': False,
            'time': float(max_death)  # conservative censoring
        })

survival_df = pd.DataFrame(survival_records).set_index('patient_id')
print(f"Survival records: {len(survival_df)}")
print(f"Events (deaths): {survival_df['event'].sum()} ({survival_df['event'].mean()*100:.1f}%)")
print(f"Censored (alive): {(~survival_df['event']).sum()}")
print(f"Time range: {survival_df['time'].min():.0f} to {survival_df['time'].max():.0f} days")

# Fix mRNA patient IDs — strip suffix (C3L-00001-02 → C3L-00001)
mrna_stripped = mrna.copy()
new_cols = []
for c in mrna.columns:
    if c.startswith('C3'):
        new_cols.append(c.rsplit('-', 1)[0])
    else:
        new_cols.append(c)
mrna_stripped.columns = new_cols

# Remove duplicate patient IDs (keep first)
mrna_stripped = mrna_stripped.T
mrna_stripped = mrna_stripped[~mrna_stripped.index.duplicated(keep='first')]

print(f"\nmRNA after stripping suffix: {mrna_stripped.shape}")

# Align patients
common = survival_df.index.intersection(mrna_stripped.index)
survival_df   = survival_df.loc[common]
mrna_aligned  = mrna_stripped.loc[common]

print(f"Common patients: {len(common)}")
print(f"Events in common: {survival_df['event'].sum()} ({survival_df['event'].mean()*100:.1f}%)")

Survival records: 208
Events (deaths): 56 (26.9%)
Censored (alive): 152
Time range: 24 to 1703 days

mRNA after stripping suffix: (231, 40796)
Common patients: 203
Events in common: 53 (26.1%)


In [5]:
import mygene

# Load our gene lists needed for the model
cox_lasso   = pickle.load(open(f'{base}/models/cox_lasso_expression.pkl', 'rb'))
gene_list   = json.load(open(f'{base}/models/gene_list.json'))
immune_cols = json.load(open(f'{base}/models/experiments/pretrain_genes.json'))

# Get our 72 Lasso-selected genes
coefs = cox_lasso.coef_[:, 0]
lasso_genes = [g for g, s in zip(gene_list, coefs != 0) if s]
print(f"Lasso genes needed: {len(lasso_genes)}")

# Convert our gene symbols to Entrez IDs
mg = mygene.MyGeneInfo()
print("Converting gene symbols to Entrez IDs...")
result = mg.querymany(lasso_genes, scopes='symbol',
                       fields='entrezgene', species='human', returnall=True)

symbol_to_entrez = {}
for r in result['out']:
    if 'entrezgene' in r:
        symbol_to_entrez[r['query']] = str(int(float(r['entrezgene'])))

print(f"Mapped: {len(symbol_to_entrez)} / {len(lasso_genes)} genes")

# Check overlap with CPTAC
mrna_entrez = set(str(g) for g in mrna_aligned.columns)
our_entrez  = set(symbol_to_entrez.values())
overlap     = our_entrez.intersection(mrna_entrez)
print(f"Our Entrez IDs in CPTAC: {len(overlap)} / {len(symbol_to_entrez)}")

# Build reverse mapping
entrez_to_symbol = {v: k for k, v in symbol_to_entrez.items()}
print(f"\nExample mappings:")
for sym, ent in list(symbol_to_entrez.items())[:5]:
    print(f"  {sym} -> {ent}")

Lasso genes needed: 72
Converting gene symbols to Entrez IDs...


3 input query terms found dup hits:	[('TRPC2', 2), ('ABCA17P', 2), ('MBL1P', 2)]
8 input query terms found no hit:	['ERO1LB', 'LST-3TM12', 'C8orf47', 'HIST1H3G', 'GPR64', 'ODZ1', 'CYP2D7P1', 'LOC654433']


Mapped: 64 / 72 genes
Our Entrez IDs in CPTAC: 64 / 64

Example mappings:
  EPGN -> 255324
  SLC47A1 -> 55244
  SIX1 -> 6495
  RHCG -> 51458
  GPC6 -> 10082


In [6]:
from sklearn.svm import NuSVR
from sklearn.preprocessing import normalize
from scipy.optimize import nnls
from lifelines import CoxPHFitter

# ── Step 1: Extract expression for our 64 genes ──────────────────
# Map Entrez IDs back to gene symbols
mrna_str = mrna_aligned.copy()
mrna_str.columns = [str(c) for c in mrna_str.columns]

# Extract our genes
our_entrez_ids = [symbol_to_entrez[g] for g in lasso_genes if g in symbol_to_entrez]
our_symbols    = [g for g in lasso_genes if g in symbol_to_entrez]

expr_cptac = mrna_str[our_entrez_ids].copy()
expr_cptac.columns = our_symbols

# Log2(TPM+1) transform — CPTAC data is raw TPM
expr_cptac = np.log2(expr_cptac.astype(float) + 1)

# Fill missing genes with 0
for gene in lasso_genes:
    col = f"{gene}_expr"
    if gene not in expr_cptac.columns:
        expr_cptac[gene] = 0.0

print(f"Expression matrix: {expr_cptac.shape}")
print(f"Value range: {expr_cptac.values.min():.2f} to {expr_cptac.values.max():.2f}")
print(f"Log2 transformed ✅")

# ── Step 2: Clinical features ─────────────────────────────────────
# No stage available in CPTAC — fill with 0
age_cptac    = pd.to_numeric(clin.set_index('PATIENT_ID').loc[common]['AGE'], 
                              errors='coerce').fillna(60)
gender_cptac = (clin.set_index('PATIENT_ID').loc[common]['SEX'] == 'Male').astype(float)

clinical_cptac = pd.DataFrame({
    'age':             age_cptac.values,
    'gender':          gender_cptac.values,
    'stage_Stage II':  0.0,  # not available
    'stage_Stage III': 0.0,  # not available
    'stage_Stage IV':  0.0   # not available
}, index=common)

print(f"\nClinical features: {clinical_cptac.shape}")
print(f"Age range: {clinical_cptac['age'].min():.0f} to {clinical_cptac['age'].max():.0f}")
print(f"Gender (male=1): {clinical_cptac['gender'].mean():.2f}")
print(f"Note: Stage features set to 0 (not available in CPTAC)")

Expression matrix: (203, 72)
Value range: 0.00 to 12.36
Log2 transformed ✅

Clinical features: (203, 5)
Age range: 35 to 81
Gender (male=1): 0.64
Note: Stage features set to 0 (not available in CPTAC)


In [7]:
from datetime import datetime

# ── Step 1: DIY CIBERSORT for immune features ─────────────────────
lm22 = pd.read_csv(f'{base}/data/external/LM22.txt', sep='\t', index_col=0)

# Need full expression for CIBERSORT — extract all LM22 genes
lm22_entrez = mg.querymany(list(lm22.index), scopes='symbol',
                            fields='entrezgene', species='human', returnall=True)

lm22_sym_to_entrez = {}
for r in lm22_entrez['out']:
    if 'entrezgene' in r:
        lm22_sym_to_entrez[r['query']] = str(int(float(r['entrezgene'])))

# Find overlap
lm22_overlap = {sym: eid for sym, eid in lm22_sym_to_entrez.items() 
                if eid in mrna_str.columns}
print(f"LM22 genes mapped to Entrez: {len(lm22_sym_to_entrez)}")
print(f"LM22 genes in CPTAC:         {len(lm22_overlap)}")
print(f"Coverage:                     {len(lm22_overlap)/len(lm22.index)*100:.1f}%")

# Extract LM22 genes from CPTAC
lm22_expr_cptac = mrna_str[[lm22_overlap[s] for s in lm22_overlap]].copy()
lm22_expr_cptac.columns = list(lm22_overlap.keys())

# Log2 transform
lm22_expr_cptac = np.log2(lm22_expr_cptac.astype(float) + 1)

# Align LM22 matrix
common_lm22 = lm22.index.intersection(lm22_expr_cptac.columns)
lm22_common = lm22.loc[common_lm22]
expr_lm22   = lm22_expr_cptac[common_lm22]

print(f"Common LM22 genes: {len(common_lm22)}")

# Run CIBERSORT
def run_cibersort_single(patient_expr, lm22_matrix):
    expr_linear = (2 ** patient_expr.values) - 1
    expr_linear = np.clip(expr_linear, 0, 1e6)
    lm22_linear = (2 ** lm22_matrix.values) - 1
    lm22_linear = np.clip(lm22_linear, 0, 1e6)
    lm22_norm   = normalize(lm22_linear, axis=0)
    expr_norm   = normalize(expr_linear.reshape(1, -1))[0]
    best_nu = 0.5; best_error = np.inf
    for nu in [0.25, 0.5, 0.75]:
        try:
            svr = NuSVR(nu=nu, kernel='linear', C=1.0)
            svr.fit(lm22_norm, expr_norm)
            error = np.mean((lm22_norm @ svr.coef_[0] - expr_norm) ** 2)
            if error < best_error:
                best_error = error; best_nu = nu
        except: continue
    try:
        svr = NuSVR(nu=best_nu, kernel='linear', C=1.0)
        svr.fit(lm22_norm, expr_norm)
        raw_weights = svr.coef_[0]
    except:
        raw_weights = np.zeros(lm22_matrix.shape[1])
    clipped = np.maximum(raw_weights, 0)
    if clipped.sum() == 0:
        clipped, _ = nnls(lm22_norm, expr_norm)
        clipped = np.maximum(clipped, 0)
    total = clipped.sum()
    final = clipped / total if total > 0 else np.ones(len(clipped)) / len(clipped)
    return dict(zip(lm22_matrix.columns, final))

print(f"\nRunning DIY CIBERSORT on {len(expr_lm22)} CPTAC patients...")
print(f"Start: {datetime.now().strftime('%H:%M:%S')}")

results_cptac = {}
for i, pid in enumerate(expr_lm22.index):
    results_cptac[pid] = run_cibersort_single(expr_lm22.loc[pid], lm22_common)
    if (i+1) % 50 == 0 or i == 0:
        print(f"  {i+1}/{len(expr_lm22)} [{datetime.now().strftime('%H:%M:%S')}]")

immune_cptac = pd.DataFrame(results_cptac).T
print(f"\nImmune features: {immune_cptac.shape}")
print(f"Row sums: {immune_cptac.sum(axis=1).mean():.4f}")
print(f"Macrophages M2: {immune_cptac['Macrophages M2'].mean():.4f}")

11 input query terms found dup hits:	[('CLCA3P', 2), ('GUSBP11', 2), ('IGHD', 2), ('IGHE', 2), ('IGHM', 2), ('IGLL3P', 2), ('TARDBPP1', 2
24 input query terms found no hit:	['ATHL1', 'C11orf80', 'CXorf57', 'EMR1', 'EMR2', 'EMR3', 'FAIM3', 'FAM198B', 'FAM212B', 'FAM65B', 'F


LM22 genes mapped to Entrez: 523
LM22 genes in CPTAC:         519
Coverage:                     94.9%
Common LM22 genes: 519

Running DIY CIBERSORT on 203 CPTAC patients...
Start: 17:42:42
  1/203 [17:42:42]
  50/203 [17:42:43]
  100/203 [17:42:45]
  150/203 [17:42:47]
  200/203 [17:42:48]

Immune features: (203, 22)
Row sums: 1.0000
Macrophages M2: 0.0510


In [8]:
# ── Step 1: Dysregulation using GTEx reference ────────────────────
# CPTAC is RNA-seq — same technology as GTEx reference ✅
gtex_ref = json.load(open(f'{base}/data/processed/gtex_reference.json'))

# Load dysregulation gene list from NB06b
dysreg_train = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
dysreg_genes = list(dysreg_train.columns)

print(f"GTEx reference genes: {len(gtex_ref)}")
print(f"Dysregulation genes needed: {len(dysreg_genes)}")

# Map dysregulation genes to Entrez
dysreg_entrez = mg.querymany(dysreg_genes, scopes='symbol',
                              fields='entrezgene', species='human', returnall=True)

dysreg_sym_to_entrez = {}
for r in dysreg_entrez['out']:
    if 'entrezgene' in r:
        dysreg_sym_to_entrez[r['query']] = str(int(float(r['entrezgene'])))

# Find available genes in CPTAC
dysreg_available = {sym: eid for sym, eid in dysreg_sym_to_entrez.items()
                    if eid in mrna_str.columns and sym in gtex_ref}

print(f"Dysreg genes mapped: {len(dysreg_sym_to_entrez)}")
print(f"Dysreg genes in CPTAC + GTEx ref: {len(dysreg_available)}")

# Compute z-scores using GTEx reference
dysreg_cptac = pd.DataFrame(index=mrna_aligned.index, 
                              columns=list(dysreg_available.keys()),
                              dtype=float)

for sym, eid in dysreg_available.items():
    tpm_vals = np.log2(mrna_str[eid].astype(float) + 1)
    gtex_mean = gtex_ref[sym]['mean']
    gtex_std  = gtex_ref[sym]['std']
    if gtex_std > 0:
        dysreg_cptac[sym] = (tpm_vals - gtex_mean) / gtex_std
    else:
        dysreg_cptac[sym] = 0.0

dysreg_cptac = dysreg_cptac.fillna(0)
print(f"\nDysregulation matrix: {dysreg_cptac.shape}")
print(f"Value range: {dysreg_cptac.values.min():.2f} to {dysreg_cptac.values.max():.2f}")
print(f"Dysregulation computed using GTEx reference ✅")

GTEx reference genes: 819
Dysregulation genes needed: 819


12 input query terms found dup hits:	[('RRN3P1', 2), ('TRPC2', 2), ('NAPSB', 2), ('ABCC6P1', 2), ('ABCA17P', 2), ('FAM95B1', 2), ('ADAM6'


Dysreg genes mapped: 819
Dysreg genes in CPTAC + GTEx ref: 817

Dysregulation matrix: (203, 817)
Value range: -2.72 to 31.45
Dysregulation computed using GTEx reference ✅


In [9]:
# Load models and scalers from leakage-free pipeline
from sksurv.ensemble import GradientBoostingSurvivalAnalysis

# Load training data to get exact feature setup
expr_train   = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg_train = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune_train = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical_tr  = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

# Align training
common_tr = expr_train.index.intersection(dysreg_train.index).intersection(
            immune_train.index).intersection(clinical_tr.index)
expr_train   = expr_train.loc[common_tr]
dysreg_train = dysreg_train.loc[common_tr]
immune_train = immune_train.loc[common_tr]
clinical_tr  = clinical_tr.loc[common_tr]

age_tr    = clinical_tr[['age']].copy()
gender_tr = (clinical_tr['gender'] == 'male').astype(float).to_frame()
stage_tr  = pd.get_dummies(clinical_tr['stage_group'], prefix='stage')
stage_tr  = stage_tr.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features_tr = pd.concat([age_tr, gender_tr, stage_tr], 
                                   axis=1).astype(float).fillna(0)

y_train = np.array(
    [(bool(e), t) for e, t in zip(clinical_tr['event'], clinical_tr['survival_time'])],
    dtype=[('event', bool), ('time', float)])

# Get Lasso genes
coefs       = cox_lasso.coef_[:, 0]
lasso_genes = [g for g, s in zip(gene_list, coefs != 0) if s]

# Expression with suffix
expr_tr_lasso = expr_train[lasso_genes].copy()
expr_tr_lasso.columns = [f"{g}_expr" for g in lasso_genes]

# Cox selection for dysregulation on full training data
print("Selecting top 20 dysregulation genes by Cox p-value...")
times  = y_train['time']
events = y_train['event']

cox_pvals_dysreg = {}
for gene in dysreg_train.columns:
    try:
        df_tmp = pd.DataFrame({'T': times, 'E': events,
                               'gene': dysreg_train[gene].values})
        cph = CoxPHFitter()
        cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
        cox_pvals_dysreg[gene] = cph.summary['p'].values[0]
    except:
        cox_pvals_dysreg[gene] = 1.0
top_dysreg_genes = list(pd.Series(cox_pvals_dysreg).nsmallest(20).index)
print(f"Top dysreg genes selected: {len(top_dysreg_genes)}")

dysreg_tr_sel = dysreg_train[top_dysreg_genes].copy()
dysreg_tr_sel.columns = [f"{g}_dysreg" for g in top_dysreg_genes]

# Interaction features for training
interactions_tr = pd.DataFrame({
    'stageIII_x_M2':   (clinical_features_tr['stage_Stage III'] *
                        immune_train['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_features_tr['stage_Stage IV'] *
                        immune_train['T cells CD8']).values,
    'age_x_stageIII':  (clinical_features_tr['age'] *
                        clinical_features_tr['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_features_tr['stage_Stage III'] *
                        immune_train['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_train['Macrophages M2'] *
                        immune_train['T cells CD8']).values,
}, index=expr_train.index)

# Build training matrix
X_train = pd.concat([expr_tr_lasso, dysreg_tr_sel,
                      immune_train, clinical_features_tr,
                      interactions_tr], axis=1).fillna(0)

print(f"Training matrix: {X_train.shape}")

# ── Build CPTAC feature matrix ────────────────────────────────────
# Expression
expr_cptac_lasso = expr_cptac[[g for g in lasso_genes 
                                if g in expr_cptac.columns]].copy()
# Fill missing genes
for g in lasso_genes:
    if g not in expr_cptac_lasso.columns:
        expr_cptac_lasso[g] = 0.0
expr_cptac_lasso = expr_cptac_lasso[lasso_genes]
expr_cptac_lasso.columns = [f"{g}_expr" for g in lasso_genes]

# Dysregulation
dysreg_cptac_sel = pd.DataFrame(index=dysreg_cptac.index)
for g in top_dysreg_genes:
    if g in dysreg_cptac.columns:
        dysreg_cptac_sel[f"{g}_dysreg"] = dysreg_cptac[g].values
    else:
        dysreg_cptac_sel[f"{g}_dysreg"] = 0.0

# Interaction features for CPTAC
# Stage is 0 so stage interactions will be 0
interactions_cptac = pd.DataFrame({
    'stageIII_x_M2':   0.0,
    'stageIV_x_CD8':   0.0,
    'age_x_stageIII':  0.0,
    'stageIII_x_Treg': 0.0,
    'M2_x_CD8':        (immune_cptac['Macrophages M2'] *
                        immune_cptac['T cells CD8']).values,
}, index=immune_cptac.index)

# Build CPTAC matrix
X_cptac = pd.concat([expr_cptac_lasso, dysreg_cptac_sel,
                      immune_cptac, clinical_cptac,
                      interactions_cptac], axis=1).fillna(0)

# Reorder to match training
X_cptac = X_cptac[X_train.columns]

print(f"CPTAC matrix:   {X_cptac.shape}")
print(f"Columns match:  {list(X_cptac.columns) == list(X_train.columns)}")
print(f"Any NaN:        {X_cptac.isna().any().any()}")

# Scale using training data
scaler     = StandardScaler()
X_train_s  = pd.DataFrame(scaler.fit_transform(X_train),
                            columns=X_train.columns, index=X_train.index)
X_cptac_s  = pd.DataFrame(scaler.transform(X_cptac),
                            columns=X_cptac.columns, index=X_cptac.index)

# Train final XGBoost on all training data
model_final = GradientBoostingSurvivalAnalysis(
    n_estimators=300, learning_rate=0.05, max_depth=2,
    min_samples_split=20, min_samples_leaf=10,
    subsample=0.8, random_state=42)
model_final.fit(X_train_s, y_train)
print(f"\nFinal model trained on {len(X_train)} patients ✅")

# Predict on CPTAC
risk_scores = model_final.predict(X_cptac_s)

# Survival labels for CPTAC
y_cptac = np.array(
    [(bool(e), float(t)) for e, t in zip(
        survival_df['event'], survival_df['time'])],
    dtype=[('event', bool), ('time', float)])

ci_cptac = concordance_index_censored(
    y_cptac['event'].astype(bool),
    y_cptac['time'],
    risk_scores)[0]

print(f"\n{'='*45}")
print(f"EXTERNAL VALIDATION RESULT (CPTAC-LUAD)")
print(f"{'='*45}")
print(f"Cohort:          CPTAC-LUAD")
print(f"Patients:        {len(y_cptac)}")
print(f"Events (deaths): {y_cptac['event'].sum()} ({y_cptac['event'].mean()*100:.1f}%)")
print(f"C-index:         {ci_cptac:.3f}")
print(f"{'='*45}")
print(f"\nFull external validation summary:")
print(f"  TCGA training:     0.702")
print(f"  GSE72094:          0.636")
print(f"  CPTAC-LUAD:        {ci_cptac:.3f}")

Selecting top 20 dysregulation genes by Cox p-value...
Top dysreg genes selected: 20
Training matrix: (478, 124)
CPTAC matrix:   (203, 124)
Columns match:  True
Any NaN:        False

Final model trained on 478 patients ✅

EXTERNAL VALIDATION RESULT (CPTAC-LUAD)
Cohort:          CPTAC-LUAD
Patients:        203
Events (deaths): 53 (26.1%)
C-index:         0.557

Full external validation summary:
  TCGA training:     0.702
  GSE72094:          0.636
  CPTAC-LUAD:        0.557
